# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import pandas as pd
import numpy as np
df = pd.read_csv("content_refresh_anonymized.csv")

df["declining_observed"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Shape:", df.shape)
print(df["declining_observed"].value_counts())

Shape: (30000, 45)
declining_observed
1    16262
0    13738
Name: count, dtype: int64


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Freshness: The paper reports that freshness effects are strongest when content is updated before it fully decays. My methodology question is: how exactly is the outcome/label for “decay” defined, and is the validation design using information available before the outcome period? Because the study is observational, the finding should be interpreted as an observed relationship rather than proof that updating content causes better performance.

Finding 2 — Striking Distance: The paper identifies positions 11–20 as an important optimization zone. My methodology question is: does the analysis control for differences in content age, search visibility, and other factors that may affect both position and performance? I would also check whether the result remains consistent across sufficiently large groups rather than relying on a small subset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest grouped validation

I re-ran the Week-5 Logistic Regression model using a grouped train/test split by client, so no client appears in both the training and test sets. The split contained 24 training clients and 8 test clients, with zero client overlap.

On the test set, the observed base rate was 51.65%. The Logistic Regression model achieved Precision@20 of 0.50 and Precision@50 of 0.62, compared with 0.30 and 0.32 for the Week-4 baseline on the same test set and metrics.

The model therefore showed higher measured precision than the baseline at both K values under the grouped split. This is evidence of useful ranking performance on unseen clients, but it should be treated as decision-support rather than proof that the model will perform the same way in future deployments.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Re-run the Week-5 model under an honest grouped split

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# -----------------------------
# 1. Define the target
# -----------------------------
y = df["declining_observed"]

# -----------------------------
# 2. Use the same features as Week 5
# -----------------------------
numeric_features = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "search_volume",
    "competition",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent"
]

features = numeric_features + categorical_features

X = df[features]
groups = df["client_id"]

# -----------------------------
# 3. Grouped split by client
# -----------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# -----------------------------
# 4. Preprocessing
# -----------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# -----------------------------
# 5. Train Logistic Regression
# -----------------------------
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# -----------------------------
# 6. Predict probabilities
# -----------------------------
test_probabilities = model.predict_proba(X_test)[:, 1]

# -----------------------------
# 7. Precision@K
# -----------------------------
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

model_p20 = precision_at_k(test_probabilities, y_test, 20)
model_p50 = precision_at_k(test_probabilities, y_test, 50)

# -----------------------------
# 8. Baseline on the SAME test set
# -----------------------------
# Week-4 baseline:
# meaningful visibility + 91+ days stale

baseline_score = (
    X_test["impressions_prev_30d"].fillna(0).values
    * (X_test["days_since_last_update"].fillna(0).values >= 91)
)

# For the baseline, use the previous-30-day visibility
# to keep the comparison aligned with the model's feature window.

baseline_p20 = precision_at_k(baseline_score, y_test, 20)
baseline_p50 = precision_at_k(baseline_score, y_test, 50)

# -----------------------------
# 9. Base rate
# -----------------------------
base_rate = y_test.mean()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

print("\nTest base rate:", round(base_rate, 4))

print("\nModel Precision@20:", round(model_p20, 3))
print("Model Precision@50:", round(model_p50, 3))

print("\nBaseline Precision@20:", round(baseline_p20, 3))
print("Baseline Precision@50:", round(baseline_p50, 3))

# -----------------------------
# 10. Comparison table
# -----------------------------
comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Logistic Regression"],
    "precision_at_20": [baseline_p20, model_p20],
    "precision_at_50": [baseline_p50, model_p50],
    "test_base_rate": [base_rate, base_rate]
})

comparison

Train shape: (22885, 12)
Test shape: (7115, 12)

Train clients: 24
Test clients: 8
Client overlap: 0

Test base rate: 0.5165

Model Precision@20: 0.5
Model Precision@50: 0.62

Baseline Precision@20: 0.3
Baseline Precision@50: 0.32


,method,precision_at_20,precision_at_50,test_base_rate
0,Week-4 baseline,0.3,0.32,0.516514
1,Logistic Regression,0.5,0.62,0.516514


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final feature set for label-derived fields and existing decision signals. `trend_direction`, `trend_pct`, `is_declining_label`, and `declining_observed` were not used as features. The Week-4 baseline outputs such as `action`, `reason_code`, and `score` were also not used.

The model uses previous-30-day performance metrics, content metadata, and other features that were available before the prediction outcome. Based on this feature check, I found no obvious label or decision-derived leakage in the final feature set.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Leakage audit

print("=== Leakage Audit ===")

# 1. Label-derived columns
label_derived = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "declining_observed"
]

print("\nLabel-derived columns:")
for col in label_derived:
    print(f"{col}: {'USED' if col in features else 'NOT USED'}")

# 2. Product / decision flags
product_flags = [
    "action",
    "reason_code",
    "score",
    "baseline_score",
    "stale_91_plus"
]

print("\nProduct / decision-derived columns:")
for col in product_flags:
    print(f"{col}: {'USED' if col in features else 'NOT USED'}")

# 3. Check final feature list
print("\nFinal features used by the model:")
for col in features:
    print("-", col)

# 4. Check for suspicious future/trend fields
suspect_words = [
    "trend",
    "declin",
    "label",
    "action",
    "reason",
    "score"
]

suspicious_features = [
    col for col in features
    if any(word in col.lower() for word in suspect_words)
]

print("\nSuspicious feature-name check:")
if suspicious_features:
    print("CHECK THESE:", suspicious_features)
else:
    print("No suspicious label/decision feature names found.")

# 5. Final verdict
print("\nLeakage verdict:")
print("PASS — the final feature set excludes label-derived trend fields and product decision flags.")
print("The model uses previous-30-day metrics, content metadata, and other information available before the prediction outcome.")

=== Leakage Audit ===

Label-derived columns:
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED
declining_observed: NOT USED

Product / decision-derived columns:
action: NOT USED
reason_code: NOT USED
score: NOT USED
baseline_score: NOT USED
stale_91_plus: NOT USED

Final features used by the model:
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- days_since_last_update
- content_age_days
- avg_position
- search_volume
- competition
- word_count
- char_count
- content_type
- main_intent

Suspicious feature-name check:
No suspicious label/decision feature names found.

Leakage verdict:
PASS — the final feature set excludes label-derived trend fields and product decision flags.
The model uses previous-30-day metrics, content metadata, and other information available before the prediction outcome.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

Original claim: The Logistic Regression model can accurately predict which content will decline.

Safer claim: On the grouped client-level test split, the Logistic Regression model showed higher measured Precision@20 and Precision@50 than the Week-4 baseline. This is an observed result on the tested data and suggests the model may provide useful decision-support for prioritizing content for review, but it does not establish that the model will perform the same way on future clients or deployments.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.